In [1]:
!pip install -q torch_geometric scikit-learn


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.6 MB/s eta 0:00:00


In [2]:
from google.colab import files

uploaded = files.upload()


Saving Labelled Yelp Dataset.csv to Labelled Yelp Dataset.csv


In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import classification_report, confusion_matrix

import torch
from torch import nn
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

DATA_PATH = "Labelled Yelp Dataset.csv"

TEXT_COL  = "Review"
LABEL_COL = "Label"

K_NEIGHBORS  = 10
MAX_FEATURES = 5000
VAL_SIZE     = 0.1
TEST_SIZE    = 0.2
RANDOM_STATE = 42


Using device: cuda


In [5]:
df = pd.read_csv(DATA_PATH)
print("Raw shape:", df.shape)
print(df.head())

df = df[[TEXT_COL, LABEL_COL]].dropna().reset_index(drop=True)
print("After dropping NA:", df.shape)

df[LABEL_COL] = df[LABEL_COL].astype(int).map({-1: 0, 1: 1})

print("Label distribution (0=fake, 1=genuine):")
print(df[LABEL_COL].value_counts())
df.head()


Raw shape: (359052, 6)
   User_id  Product_id  Rating       Date  \
0      923           0       3  12/8/2014   
1      924           0       3  5/16/2013   
2      925           0       4   7/1/2013   
3      926           0       4  7/28/2011   
4      927           0       4  11/1/2010   

                                              Review  Label  
0  The food at snack is a selection of popular Gr...     -1  
1  This little place in Soho is wonderful. I had ...     -1  
2  ordered lunch for 15 from Snack last Friday. Â...     -1  
3  This is a beautiful quaint little restaurant o...     -1  
4  Snack is great place for a Â casual sit down l...     -1  
After dropping NA: (359052, 2)
Label distribution (0=fake, 1=genuine):
Label
1    322167
0     36885
Name: count, dtype: int64


,Review,Label
0,The food at snack is a selection of popular Gr...,0
1,This little place in Soho is wonderful. I had ...,0
2,ordered lunch for 15 from Snack last Friday. Â...,0
3,This is a beautiful quaint little restaurant o...,0
4,Snack is great place for a Â casual sit down l...,0


In [7]:
df = pd.read_csv(DATA_PATH)
df.head()

,User_id,Product_id,Rating,Date,Review,Label
0,923,0,3,12/8/2014,The food at snack is a selection of popular Gr...,-1
1,924,0,3,5/16/2013,This little place in Soho is wonderful. I had ...,-1
2,925,0,4,7/1/2013,ordered lunch for 15 from Snack last Friday. Â...,-1
3,926,0,4,7/28/2011,This is a beautiful quaint little restaurant o...,-1
4,927,0,4,11/1/2010,Snack is great place for a Â casual sit down l...,-1


In [8]:
indices = np.arange(len(df))
labels  = df[LABEL_COL].values

train_idx, test_idx = train_test_split(
    indices,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=labels,
)

train_labels = labels[train_idx]
train_idx, val_idx = train_test_split(
    train_idx,
    test_size=VAL_SIZE / (1 - TEST_SIZE),
    random_state=RANDOM_STATE,
    stratify=train_labels,
)

print("Train size:", len(train_idx))
print("Val size:  ", len(val_idx))
print("Test size: ", len(test_idx))


Train size: 251335
Val size:   35906
Test size:  71811


In [ ]:
vectorizer = TfidfVectorizer(
    max_features=MAX_FEATURES,
    ngram_range=(1, 2),
    stop_words="english",
)

X = vectorizer.fit_transform(df[TEXT_COL].astype(str).values)
print("TF-IDF shape:", X.shape)

X_dense = torch.tensor(X.toarray(), dtype=torch.float)
y = torch.tensor(df[LABEL_COL].values, dtype=torch.long)

num_nodes, num_features = X_dense.shape
print("Num nodes:", num_nodes, "Num features:", num_features)


TF-IDF shape: (359052, 5000)


In [1]:
nn_model = NearestNeighbors(
    n_neighbors=K_NEIGHBORS + 1,
    metric="cosine"
)
nn_model.fit(X)

distances, indices_knn = nn_model.kneighbors(X)
print("kNN indices shape:", indices_knn.shape)

edge_index_list = []
for i in range(num_nodes):
    neighbors = indices_knn[i, 1:]
    for j in neighbors:
        edge_index_list.append([i, j])
        edge_index_list.append([j, i])

edge_index = torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()
print("edge_index shape:", edge_index.shape)


NameError: name 'NearestNeighbors' is not defined